## Desafio 2 — Consumo de Dados (SWAPI) | Globo (Engenharia de Dados)

Este notebook implementa um pipeline simples de **extração → transformação → carga (ETL)** a partir da API pública do Star Wars (SWAPI), armazenando os dados em **SQLite** e gerando **insights via SQL**.

**Stack utilizada**
- Python (requests, pandas)
- SQLite (armazenamento)
- SQL (análises e insights)
- Notebook (Jupyter/Colab)

### 1. Bibliotecas, variáveis e configurações

Nesta seção são importadas as bibliotecas necessárias e definidos os endpoints que serão consumidos.

In [2]:
# Bibliotecas
import sqlite3
import time
from pathlib import Path

import pandas as pd
import requests

In [3]:
# Variáveis
url = 'https://swapi.dev/api'
endpoint = ['films', 'people', 'planets', 'species', 'starships', 'vehicles']

### 2. Funções auxiliares (extração com paginação e extração de ID)

Extração Paginada (`fetch_all`): Esta função percorre todos os links next de um endpoint até consolidar todos os registros em uma única lista, evitando perda de dados.

Tratamento de IDs (`extract_id`): Esta função é responsável por extrair somente o identificador numérico final das URLs para garantir o relacionamento entre os registros.

In [4]:
def fetch_all(base_url: str, endpoint: str, sleep_s: float = 0.1) -> list[dict]:
    """
    Consome dados da API de forma paginada.
    
    Args:
        base_url: URL base da API (ex: https://swapi.dev/api).
        endpoint: O recurso específico a ser consumido (ex: 'people', 'starships').
        sleep_s: Tempo de espera entre requisições para evitar rate limit.
        
    Returns:
        Uma lista contendo todos os registros (dicts) encontrados no endpoint.
    """
    # Monta a URL inicial combinando a base e o endpoint
    next_url = f"{base_url}/{endpoint}/"
    out = []

    # Itera enquanto houver uma URL de próxima página retornada pela API
    while next_url:
        # Realiza a requisição GET com um limite de tempo (timeout) para evitar travamentos
        r = requests.get(next_url, timeout=30)
        
        # Garante que a função pare caso ocorra um erro de conexão ou permissão (4xx ou 5xx)
        r.raise_for_status()
        
        data = r.json()
        
        # Adiciona os resultados da página atual à lista
        out.extend(data["results"])
        
        # Atualiza a URL para a próxima página ou None, caso chegue ao fim
        next_url = data["next"]
        
        # Pausa controlada para evitar sobrecarga da API
        time.sleep(sleep_s)

    return out

# Função auxiliar para extrair o ID da URL
def extract_id(url_string):
    """
    Extrai o identificador numérico final de uma URL da SWAPI.
    
    Transforma strings no formato 'https://swapi.dev/api/people/1/' em inteiros (1).
    Essencial para a criação de Chaves Primárias (PK) e Estrangeiras (FK) no banco de dados.
    """
    if not url_string: 
        return None
    
    # Extrai o número entre as últimas barras da URL
    return int(url_string.split('/')[-2])

### 3. Extração

Nesta etapa, os dados são consumidos diretamente da API e mantidos no formato próximo ao original (JSON → DataFrame), com mínima transformação.

In [5]:
# Dicionário para centralizar todos os dados extraídos da API
data = {}

# Iterar sobre todos os endpoints da lista
for item in endpoint:
    data[item] = fetch_all(url, item)

# Armazenar todos os registros em suas devidas variáveis
films_json = data["films"]
people_json = data["people"]
planets_json = data["planets"]
starships_json = data["starships"]

### 4. Transformação e limpeza dos dados 
Nesta etapa, os dados brutos são estruturados em DataFrames e preparados para o modelo relacional. O foco é garantir a qualidade e a tipagem correta para o consumo analítico.

Normalização: Criação da tabela de ligação film_characters para gerenciar o relacionamento entre filmes e personagens.

Limpeza dos dados: Tratamento de campos numéricos que contêm strings como unknown, n/a, ou caracteres especiais (ex: vírgulas e unidades de medida), garantindo que métricas como length sejam puramente numéricas.

Tipagem: Conversão de colunas críticas para int64, otimizando a performance de cálculos e agregados no SQL.

Seleção de Atributos: Filtragem apenas das colunas relevantes para a área de negócio, reduzindo a carga e o uso de memória.


In [7]:
# 1 - Dataframe (Filmes)
films_df = pd.DataFrame(films_json)[
    ["url", "title", "episode_id", "director", "producer"]
].copy()

# Extração dos IDs dos filmes
films_df['film_id'] = films_df['url'].apply(extract_id)

# Selecionar somente as colunas necessárias
films_df = films_df[["film_id", "title", "episode_id", "director", "producer"]]

# 2 - Dataframe (Personagens que participaram dos filmes)
rows = []
for f in films_json:
    # Extrai o ID do filme uma única vez por iteração do loop
    film_id = extract_id(f["url"]) 
    
    for person_url in f.get("characters", []):
        # Extrai o ID do personagem para cada item da lista
        rows.append({
            "film_id": film_id, 
            "person_id": extract_id(person_url)
        })

film_characters_df = pd.DataFrame(rows)

# 3 - Dataframe (Personagens)
people_df = pd.DataFrame(people_json)[
    ["url", "name", "height", "mass", "gender", "birth_year", "homeworld"]
].copy()

# Tratamento para extrair somente o id do URL
people_df['person_id'] = people_df['url'].apply(extract_id)

# Seleciona somente as colunas necessárias
people_df = people_df[["person_id", "name", "height", "mass", "gender", "birth_year", "homeworld"]]

# 4 - Dataframe (Planetas)
planets_df = pd.DataFrame(planets_json)[
    ["url", "name", "climate", "population", "terrain"]
].copy()

# Tratamento para extrair somente o id do URL
planets_df['planet_id'] = planets_df['url'].apply(extract_id)

# Seleciona somente as colunas necessárias
planets_df = planets_df[["planet_id", "name", "climate", "population", "terrain"]]

# 5 - Dataframe (Naves Espaciais)
starships_df = pd.DataFrame(starships_json)[
    ["url", "name", "model", "starship_class", "max_atmosphering_speed", "length"]
].copy()

# Tratamento para extrair somente o id do URL
starships_df['starship_id'] = starships_df['url'].apply(extract_id)

# Seleciona somente as colunas necessárias
starships_df = starships_df[["starship_id", "name", "model", "starship_class", "max_atmosphering_speed", "length"]]

# Filtra os valores conhecidos da API que não são numéricos
starships_df = starships_df[
    ~starships_df['max_atmosphering_speed'].isin(['n/a', 'unknown'])
]

# Remove letras e caracteres especiais (ex: "1000km" -> "1000")
starships_df['max_atmosphering_speed'] = (
    starships_df['max_atmosphering_speed']
    .str.replace(r'[^0-9]', '', regex=True)
)

# Remove vírgulas (comum em números grandes na SWAPI como '1,600')
starships_df['length'] = starships_df['length'].str.replace(',', '', regex=False)


# Remove o que sobrou de vazio antes de converter para inteiro
starships_df = starships_df[starships_df['max_atmosphering_speed'] != '']

# Conversão final
starships_df['max_atmosphering_speed'] = starships_df['max_atmosphering_speed'].astype('int64')
starships_df['length'] = starships_df['length'].astype('float64')

## 5. Persistência no SQLite

Os DataFrames são persistidos no SQLite para permitir:
- Execução de queries SQL de forma reprodutível
- Joins relacionais
- Geração de insights com rastreabilidade

In [8]:
# Carregamento dos dados
db_path = "data/swapi.db"
Path("data").mkdir(exist_ok=True)

conn = sqlite3.connect(db_path)

films_df.to_sql("films", conn, if_exists="replace", index=False)
film_characters_df.to_sql("film_characters", conn, if_exists="replace", index=False)
people_df.to_sql("people", conn, if_exists="replace", index=False)
planets_df.to_sql("planets", conn, if_exists="replace", index=False)
starships_df.to_sql("starships", conn, if_exists="replace", index=False)

# Validação de quantidade de registros
print("films:", len(films_df))
print("film_characters:", len(film_characters_df))
print("people:", len(people_df))
print("planets:", len(planets_df))
print("starships:", len(starships_df))

films: 6
film_characters: 162
people: 82
planets: 60
starships: 29


### Insight 1 - Qual personagem apareceu em mais filmes?

**Definição do problema**  
Identificar o personagem com maior número de aparições em filmes dos Star Wars.

**Como foi calculado**  
- A tabela `film_characters` representa o relacionamento (filme ↔ personagem).
- Foi realizado a contagem de quantas vezes cada `person_id` aparece em `film_characters`.
- Através do JOIN com `people` para obter o nomes.

In [9]:
query_people = """
SELECT 
    p.name,
    COUNT(*) AS total_filmes
FROM film_characters fc
JOIN people p
    ON p.person_id = fc.person_id
GROUP BY p.name
ORDER BY total_filmes DESC
LIMIT 1;
"""

people_result = pd.read_sql_query(query_people, conn)
print(f'personagem com o maior número de aparições é: {people_result['name'].values[0]}\ncom o total de: {people_result['total_filmes'].values[0]} aparições')

personagem com o maior número de aparições é: R2-D2
com o total de: 6 aparições


### Insight 2 - Quais são os planetas mais quentes?

**Limitação dos dados**  
A SWAPI não fornece temperatura numérica. Portanto, este insight utiliza o campo `climate` como proxy.

**Critério adotado (proxy de calor)**  
Foi criado um `heat_score` baseado na descrição climática:
- `hot` → score 3
- `arid` → score 2
- `tropical` → score 1
- outros → score 0

In [11]:
query_planet = """
SELECT 
    name,
    climate,
    CASE 
        WHEN climate LIKE '%hot%' THEN 3      -- Prioridade máxima: se tiver 'hot', não importa o resto.
        WHEN climate LIKE '%arid%' THEN 2     -- Segunda prioridade: se não for 'hot', mas for 'arid'.
        WHEN climate LIKE '%tropical%' THEN 1 -- Terceira prioridade: climas tropicais.
    ELSE 0
    END AS heat_score
FROM planets
WHERE heat_score > 0;
"""

planet_result = pd.read_sql_query(query_planet, conn)
planet_result.sort_values(by=['heat_score'], ascending=False)

,name,climate,heat_score
4,Mustafar,hot,3
6,Felucia,"hot, humid",3
7,Saleucami,hot,3
8,Rodia,hot,3
0,Tatooine,arid,2
2,Geonosis,"temperate, arid",2
3,Utapau,"temperate, arid, windy",2
9,Trandosha,arid,2
10,Socorro,arid,2
11,Malastare,"arid, temperate, tropical",2


### Insight 3 - Quais são as naves espaciais mais rápidas?

**Definição do problema**  
Identificar as naves espaciais mais rápidas do Star Was.

**Como foi calculado**  
Foi utilizado `max_atmosphering_speed` como métrica principal.

In [12]:
query_starship = """
    SELECT
        ROW_NUMBER() OVER (ORDER BY max_atmosphering_speed DESC) AS ranking,
        name,
        model,
       max_atmosphering_speed
    FROM starships;
"""

starships_result = pd.read_sql_query(query_starship, conn)
starships_result.sort_values(by=['ranking'], ascending=True)

,ranking,name,model,max_atmosphering_speed
0,1,H-type Nubian yacht,H-type Nubian yacht,8000
1,2,J-type diplomatic barge,J-type diplomatic barge,2000
2,3,Theta-class T-2c shuttle,Theta-class T-2c shuttle,2000
3,4,Solar Sailer,Punworcca 116-class interstellar sloop,1600
4,5,Jedi Interceptor,Eta-2 Actis-class light interceptor,1500
5,6,A-wing,RZ-1 A-wing Interceptor,1300
6,7,TIE Advanced x1,Twin Ion Engine Advanced x1,1200
7,8,Scimitar,Star Courier,1180
8,9,Jedi starfighter,Delta-7 Aethersprite-class interceptor,1150
9,10,Naboo fighter,N-1 starfighter,1100


### Insight 4 — Qual é a “arma” mais poderosa do universo de Star Wars?

**Limitação dos dados**  
A SWAPI não possui um endpoint específico de “weapons” nem um atributo explícito de poder destrutivo.

**Critério adotado (proxy de poder)**  
Para responder de forma objetiva, utilizei um proxy baseado em métricas disponíveis no dataset.  
Neste notebook, foi utilizado **`length` (comprimento)** da nave como aproximação de poder/escala bélica.

Racional:
- estruturas muito grandes tendem a ter maior capacidade destrutiva no universo Star Wars (ex.: estações de batalha, Death Star e etc)
- `length` é um atributo numérico e permite ranking reproduzível

> Observação: esse insight é uma aproximação baseada nos dados disponíveis na SWAPI.

In [13]:
query_weapon = """
    SELECT
        name,
        starship_class,
        length
    FROM starships
    WHERE length IS NOT NULL
    ORDER BY length DESC
    LIMIT 1;
"""

weapon_result = pd.read_sql_query(query_weapon, conn)
weapon_result

,name,starship_class,length
0,Star Destroyer,Star Destroyer,1600.0


In [ ]:
# Fechar conexão com o banco de dados
conn.close()